# 01 · Limpeza dos dados

**Projeto:** Previsão de preços de apartamentos em Lisboa com Machine Learning

Este é o primeiro passo do projeto. Os dados vêm de uma exportação de anúncios imobiliários e trazem **muita coisa que não interessa** (fotos, vídeos, contactos…) e alguns **problemas** (duplicados, valores impossíveis, campos em texto que deviam ser números).

O objetivo deste notebook é transformar essa exportação numa tabela **limpa, pequena e fiável**, pronta para análise e modelação.

| | |
|---|---|
| **Entrada** | `data/raw/dataset_*.csv` (a exportação mais recente) |
| **Saída** | `data/processed/dados_limpos.csv` |

> 🔒 **Privacidade:** os dados (brutos e limpos) **não são publicados** no GitHub — estão excluídos pelo `.gitignore`. Por isso, ao longo dos notebooks mostramos apenas **resumos e estatísticas**, nunca anúncios individuais.

### Como ler este notebook
- As **células de texto** (como esta) explicam *o quê* e *porquê* de cada passo.
- As **células de código** têm comentários (linhas começadas por `#`) que explicam *como* — o que faz cada instrução.
- Por baixo de cada célula de código aparece o **resultado**, seguido de uma breve **leitura** do que significa.

### Passos
1. Ler a exportação mais recente
2. Escolher as colunas úteis (e deixar de fora dados pessoais)
3. Primeira inspeção: tipos de dados e valores em falta
4. Garantir o âmbito: apartamentos à venda
5. Remover valores impossíveis
6. Remover duplicados
7. Converter o andar para número
8. Converter os campos sim/não para 1/0
9. Criar o preço por m²
10. Gravar e resumir

## 0. Preparação

Importamos as bibliotecas e a configuração partilhada do projeto (`src/utils.py`), onde estão definidas as pastas, as cores dos gráficos e outros parâmetros comuns a todos os notebooks.

In [1]:
# --- Bibliotecas -------------------------------------------------------------
import sys                 # acesso a definições do Python (usado para encontrar a pasta 'src')
from pathlib import Path   # para trabalhar com caminhos de pastas/ficheiros

import pandas as pd        # a biblioteca principal para tabelas de dados (DataFrames)

# --- Configuração partilhada do projeto ---------------------------------------
# Procurar a pasta do projeto: começamos na pasta atual (Path.cwd()) e subimos
# pasta a pasta (.parent) até encontrar a pasta 'src'. Assim o notebook funciona
# quer seja aberto a partir da pasta 'notebooks/', quer a partir da raiz do projeto.
RAIZ_PROJETO = Path.cwd()
while not (RAIZ_PROJETO / "src").is_dir() and RAIZ_PROJETO != RAIZ_PROJETO.parent:
    RAIZ_PROJETO = RAIZ_PROJETO.parent
sys.path.append(str(RAIZ_PROJETO))   # agora o Python consegue encontrar o módulo 'src'
from src.utils import DADOS_RAW, DADOS_PROCESSADOS, FICHEIRO_LIMPO, num_pt

# Mostrar até 30 colunas quando imprimimos uma tabela (por defeito o pandas esconde algumas)
pd.set_option("display.max_columns", 30)

## 1. Ler a exportação mais recente

Procuramos automaticamente o ficheiro `dataset_*.csv` **mais recente** na pasta `data/raw/`. Assim, quando exportarmos dados novos, basta colocá-los nessa pasta e voltar a correr os notebooks — sem mudar nenhum nome no código.

In [2]:
# .glob("dataset_*.csv") procura todos os ficheiros cujo nome começa por "dataset_"
# e acaba em ".csv" (o * significa "qualquer coisa").
# sorted(..., key=...) ordena-os pela data de modificação (st_mtime), do mais
# antigo para o mais recente.
ficheiros = sorted(DADOS_RAW.glob("dataset_*.csv"), key=lambda f: f.stat().st_mtime)

# Se a lista estiver vazia, paramos com uma mensagem clara a explicar o que falta
if not ficheiros:
    raise FileNotFoundError("Coloca a exportação de anúncios em data/raw/ (ficheiro 'dataset_*.csv').")

ficheiro_raw = ficheiros[-1]   # [-1] = último elemento da lista = o mais recente

# pd.read_csv lê o ficheiro CSV para uma tabela (DataFrame).
# low_memory=False evita avisos quando há colunas com tipos misturados.
raw = pd.read_csv(ficheiro_raw, low_memory=False)

print("Ficheiro lido: a exportação mais recente da pasta data/raw/")
# .shape devolve (nº de linhas, nº de colunas)
print(f"Linhas (anúncios): {raw.shape[0]}")
print(f"Colunas:           {raw.shape[1]}")

Ficheiro lido: a exportação mais recente da pasta data/raw/
Linhas (anúncios): 1661
Colunas:           453


São **mais de 450 colunas**! A maioria são listas "achatadas": cada foto do anúncio dá origem a colunas como `multimedia/images/0/url`, `multimedia/images/1/url`, etc.

Vamos agrupar as colunas pelo prefixo para perceber o que há:

In [3]:
# raw.columns é a lista de nomes das colunas.
# .str.split("/").str[0] fica só com a parte antes da primeira "/"
#   (ex.: "multimedia/images/0/url" → "multimedia").
# .value_counts() conta quantas vezes aparece cada prefixo.
grupos = raw.columns.str.split("/").str[0].value_counts()

# Mostramos os 12 grupos com mais colunas
grupos.head(12).to_frame("nº de colunas")

,nº de colunas
multimedia,366
contactInfo,14
features,5
priceInfo,5
labels,4
parkingSpace,3
detailedType,2
ribbons,2
dropDate,1
bathrooms,1


Confirma-se: quase todas as colunas são de **imagens (`multimedia`)**, **contactos do anunciante (`contactInfo`)** e **etiquetas visuais**. Nada disto ajuda a prever o preço.

## 2. Escolher as colunas úteis

Ficamos apenas com as colunas que **descrevem o imóvel** e damos-lhes nomes em português, mais fáceis de ler.

Decisões importantes:
- **Dados pessoais ficam de fora:** telefones e nomes dos anunciantes nunca são copiados.
- **`municipality` → `freguesia`:** na exportação, para Lisboa, este campo contém a freguesia (ex.: Arroios, Estrela).
- **`descida_preco_pct`** e **`url`** ficam guardados para consulta, mas **não** vão ser usados para prever (explicado no notebook 03).

In [4]:
# Dicionário no formato {"nome original": "nome novo"}
colunas = {
    "propertyCode": "id_anuncio",
    "price": "preco",
    "size": "area_m2",
    "rooms": "quartos",
    "bathrooms": "casas_banho",
    "floor": "andar",
    "hasLift": "elevador",
    "status": "estado",
    "propertyType": "tipo",
    "newDevelopment": "empreendimento_novo",
    "municipality": "freguesia",
    "district": "bairro",
    "latitude": "latitude",
    "longitude": "longitude",
    "features/hasTerrace": "terraco",
    "features/hasSwimmingPool": "piscina",
    "features/hasGarden": "jardim",
    "features/hasAirConditioning": "ar_condicionado",
    "features/hasBoxRoom": "arrecadacao",
    "parkingSpace/hasParkingSpace": "estacionamento",
    "priceDropPercentage": "descida_preco_pct",
    "operation": "operacao",
    "url": "url",
}
# raw[list(colunas)] seleciona só as colunas originais que estão no dicionário;
# .rename(columns=colunas) troca os nomes pelos nomes em português.
df = raw[list(colunas)].rename(columns=colunas)
print(f"Ficámos com {df.shape[1]} colunas (de {raw.shape[1]}).")

Ficámos com 23 colunas (de 453).


## 3. Primeira inspeção

Antes de mexer nos dados, vemos **que tipo de informação** cada coluna tem e **quanta informação falta**.

In [5]:
# Construímos uma pequena tabela-resumo com uma linha por coluna:
#  - df.dtypes: o tipo de dados de cada coluna (int64 = inteiro, float64 = decimal,
#               object = texto ou misto, bool = verdadeiro/falso)
#  - df.isna().mean(): isna() marca True onde falta valor; a média de True/False
#               dá a proporção de valores em falta (× 100 para ficar em %)
#  - df.nunique(): quantos valores diferentes existem em cada coluna
inspecao = pd.DataFrame({
    "tipo de dados": df.dtypes.astype(str),
    "valores em falta (%)": (df.isna().mean() * 100).round(1),
    "valores diferentes": df.nunique(),
})

# Ordenamos da coluna com mais valores em falta para a com menos
inspecao.sort_values("valores em falta (%)", ascending=False)

,tipo de dados,valores em falta (%),valores diferentes
descida_preco_pct,float64,89.2,16
estacionamento,object,54.6,1
piscina,object,45.6,2
jardim,object,43.0,2
terraco,object,40.6,2
arrecadacao,object,37.3,2
andar,object,16.9,25
ar_condicionado,object,9.8,2
bairro,object,7.4,90
elevador,object,6.0,2


**O que retiramos daqui:**
- **Preço, área, quartos, casas de banho e localização estão sempre preenchidos** — ótimo, são as variáveis mais importantes.
- **Piscina, jardim, terraço e arrecadação faltam em ~40% dos anúncios.** Isto não quer dizer "não tem": quer dizer que o anunciante não indicou.
- **Estacionamento falta em ~55%** — mas aqui há uma particularidade (ver passo 8).
- **`andar`** está guardado como **texto** (`object`), porque inclui valores como `bj` (rés-do-chão). Vamos convertê-lo para número.

E um resumo estatístico das variáveis numéricas principais:

In [6]:
# .describe() calcula estatísticas básicas: contagem, média (mean), desvio-padrão (std),
# mínimo, quartis (25%, 50% = mediana, 75%) e máximo.
df[["preco", "area_m2", "quartos", "casas_banho"]].describe().round(0)

,preco,area_m2,quartos,casas_banho
count,1661.0,1661.0,1661.0,1661.0
mean,982367.0,135.0,3.0,2.0
std,799188.0,83.0,1.0,1.0
min,139000.0,11.0,0.0,0.0
25%,530000.0,82.0,2.0,1.0
50%,735000.0,116.0,2.0,2.0
75%,1150000.0,165.0,3.0,3.0
max,10000000.0,799.0,10.0,7.0


Há sinais de valores estranhos nos extremos (por exemplo, o mínimo e o máximo da área e dos quartos). Tratamos disso no passo 5.

## 4. Âmbito: apartamentos à venda

O projeto foca-se em **apartamentos à venda**. A exportação já deve vir filtrada, mas confirmamos — é uma proteção caso uma exportação futura traga moradias ou arrendamentos por engano.

Tipos considerados apartamento: `flat` (apartamento), `duplex`, `penthouse` (cobertura) e `studio` (estúdio).

In [7]:
# Lista onde vamos registando quantas linhas restam depois de cada passo.
# No fim, transforma-se num "funil" que mostra o que foi removido e porquê.
contagem = [("Exportação original", len(df))]   # len(df) = nº de linhas

# Quantos anúncios há de cada tipo e de cada operação (venda/arrendamento)
print(df["tipo"].value_counts().to_string(), "\n")
print(df["operacao"].value_counts().to_string())

TIPOS_APARTAMENTO = ["flat", "duplex", "penthouse", "studio"]

# FILTRAR LINHAS: df[condição] fica só com as linhas onde a condição é verdadeira.
#  - .isin(lista): verdadeiro se o valor estiver na lista
#  - & significa "E" (as duas condições têm de ser verdadeiras)
df = df[df["tipo"].isin(TIPOS_APARTAMENTO) & (df["operacao"] == "sale")]

# A coluna "operacao" já não é precisa (agora é sempre "sale"), por isso removemo-la
df = df.drop(columns="operacao")
contagem.append(("Só apartamentos à venda", len(df)))

tipo
flat         1466
duplex        104
penthouse      46
studio         45 

operacao
sale    1661


## 5. Valores impossíveis ou muito suspeitos

Erros de introdução de dados acontecem (ex.: um "T60" ou um apartamento com 5 m²). Usamos **limites de bom senso**:

- Área entre **20 e 600 m²**
- No máximo **8 quartos**

Estes limites são deliberadamente largos: só queremos apanhar erros óbvios, não casas invulgares mas reais.

In [8]:
AREA_MIN, AREA_MAX = 20, 600
QUARTOS_MAX = 8

# .between(a, b) é verdadeiro quando o valor está entre a e b.
# O "~" inverte: verdadeiro quando está FORA do intervalo.
fora_area = ~df["area_m2"].between(AREA_MIN, AREA_MAX)
fora_quartos = df["quartos"] > QUARTOS_MAX

# .sum() numa coluna de True/False conta quantos True existem
print(f"Anúncios com área fora de {AREA_MIN}–{AREA_MAX} m²: {fora_area.sum()}")
print(f"Anúncios com mais de {QUARTOS_MAX} quartos:        {fora_quartos.sum()}")

# Ficamos com as linhas que NÃO estão fora da área E NÃO têm quartos a mais
df = df[~fora_area & ~fora_quartos]
contagem.append(("Sem valores impossíveis", len(df)))

Anúncios com área fora de 20–600 m²: 7
Anúncios com mais de 8 quartos:        2


## 6. Duplicados

Este é um dos passos **mais importantes** — e mais traiçoeiros.

A mesma casa pode aparecer várias vezes:
- **a)** O mesmo anúncio repetido na exportação (mesmo código).
- **b)** A mesma casa publicada por **duas agências**, ou **frações iguais** no mesmo empreendimento (mesma área, mesmo preço, mesmo prédio).

**Porque é que isto importa?** Se a mesma casa ficar duas vezes nos dados, pode calhar uma cópia no treino e outra no teste. O modelo "reconhece" a casa e acerta — não porque aprendeu, mas porque **decorou**. A avaliação pareceria melhor do que é na realidade.

Para o caso b), consideramos duplicado quando coincidem: **localização** (arredondada a ~100 m), **área**, **quartos**, **casas de banho** e **preço**.

In [9]:
# a) Mesmo código de anúncio
n_antes = len(df)
# drop_duplicates remove linhas repetidas; subset diz quais colunas comparar.
# Quando há repetições, fica a primeira e as restantes são removidas.
df = df.drop_duplicates(subset="id_anuncio")
print(f"a) Anúncios repetidos (mesmo código): {n_antes - len(df)}")

# b) Mesma casa com códigos diferentes
n_antes = len(df)
# Criamos uma "chave" com as características que identificam uma casa.
# .round(3) arredonda as coordenadas a 3 casas decimais (≈ 100 metros), para
# apanhar a mesma casa mesmo que cada agência a localize num ponto ligeiramente diferente.
chave = (df.assign(lat=df["latitude"].round(3), lon=df["longitude"].round(3))
           [["lat", "lon", "area_m2", "quartos", "casas_banho", "preco"]])

# .duplicated() marca True nas linhas cuja chave já apareceu antes;
# com o "~" ficamos só com as primeiras ocorrências.
df = df[~chave.duplicated()]
print(f"b) Mesma casa publicada mais de uma vez: {n_antes - len(df)}")

contagem.append(("Sem duplicados", len(df)))

a) Anúncios repetidos (mesmo código): 0
b) Mesma casa publicada mais de uma vez: 29


## 7. Converter o andar para número

O andar vem como texto. A maioria são números (`"1"`, `"2"`…), mas há códigos especiais:

| Código | Significado | Valor atribuído |
|---|---|---|
| `bj` | rés-do-chão | 0 |
| `en` | entresol | 0,5 |
| `st`, `ss` | semicave / subcave | −1 |

Quando o andar não vem no anúncio, fica **vazio** (`NaN`). Não inventamos valores — o modelo vai lidar com isso mais tarde.

In [10]:
print("Valores originais mais frequentes:")
print(df["andar"].value_counts(dropna=False).head(8).to_string())

CODIGOS_ANDAR = {"bj": 0, "en": 0.5, "st": -1, "ss": -1}

# 1) .replace(dicionário) troca os códigos de texto pelos números da tabela acima
# 2) pd.to_numeric converte o texto ("1", "2"...) em números;
#    errors="coerce" transforma em vazio (NaN) qualquer valor que não consiga converter
df["andar"] = pd.to_numeric(df["andar"].replace(CODIGOS_ANDAR), errors="coerce")

print(f"\nDepois da conversão: tipo = {df['andar'].dtype}, em falta = {num_pt(df['andar'].isna().mean()*100)}%")

Valores originais mais frequentes:
andar
1      276
NaN    275
2      235
3      199
bj     172
4      141
5      103
6       61

Depois da conversão: tipo = float64, em falta = 16,9%


## 8. Converter os campos sim/não para 1/0

Os modelos trabalham com números, por isso `True/False` passa a `1/0`.

Há uma distinção importante sobre os **valores em falta**:
- **Estacionamento:** a exportação só preenche este campo quando a casa **tem** estacionamento (nos dados não existe nenhum `False`). Por isso, vazio = **não tem** → `0`.
- **Restantes** (terraço, piscina, jardim, elevador…): existem `True` e `False` nos dados, portanto vazio significa **"não sabemos"**. Fica vazio — não inventamos.

In [11]:
print("Valores no campo 'estacionamento' antes da conversão:",
      df["estacionamento"].value_counts(dropna=False).to_dict())

SIM_NAO = ["elevador", "empreendimento_novo", "terraco", "piscina", "jardim",
           "ar_condicionado", "arrecadacao", "estacionamento"]
# Ciclo "for": repete a mesma operação para cada coluna da lista.
# .map(dicionário) troca cada valor pelo correspondente no dicionário.
# Valores que não estão no dicionário (os vazios) continuam vazios.
for coluna in SIM_NAO:
    df[coluna] = df[coluna].map({True: 1, False: 0, "True": 1, "False": 0})

# Só no estacionamento: vazio significa "não tem" → preenchemos com 0
df["estacionamento"] = df["estacionamento"].fillna(0)

# Descida de preço: vazio = o preço nunca baixou
df["descida_preco_pct"] = df["descida_preco_pct"].fillna(0)

# A média de uma coluna 0/1 é a proporção de "sim" (os vazios são ignorados)
(df[SIM_NAO].mean() * 100).round(0).to_frame("% com 'sim' (entre os conhecidos)")

Valores no campo 'estacionamento' antes da conversão: {nan: 890, True: 733}


,% com 'sim' (entre os conhecidos)
elevador,68.0
empreendimento_novo,21.0
terraco,33.0
piscina,20.0
jardim,24.0
ar_condicionado,52.0
arrecadacao,37.0
estacionamento,45.0


## 9. Preço por m²

Criamos o **preço por m²**, muito útil para comparar casas de tamanhos diferentes na análise exploratória.

⚠️ **Atenção:** esta variável **não pode ser usada para prever o preço**, porque é calculada a partir do próprio preço. Seria como dar ao aluno as respostas do exame.

In [12]:
# Operações entre colunas são feitas linha a linha: o preço de cada casa
# é dividido pela área dessa mesma casa.
df["preco_m2"] = (df["preco"] / df["area_m2"]).round(0)
df["preco_m2"].describe().round(0)

count     1623.0
mean      7202.0
std       2429.0
min       2135.0
25%       5564.0
50%       6805.0
75%       8333.0
max      25750.0
Name: preco_m2, dtype: float64

## 10. Gravar e resumir

Gravamos o resultado em `data/processed/dados_limpos.csv`. Usamos a codificação `utf-8-sig` para que os acentos apareçam bem quando o ficheiro é aberto no Excel.

In [13]:
# Cria a pasta de destino, caso ainda não exista
DADOS_PROCESSADOS.mkdir(parents=True, exist_ok=True)

# Grava a tabela em CSV. index=False evita gravar a coluna de numeração das linhas.
df.to_csv(FICHEIRO_LIMPO, index=False, encoding="utf-8-sig")

# Transformamos a lista "contagem" numa tabela (o funil da limpeza).
# .diff() calcula a diferença para a linha anterior; o "-" torna-a positiva
# (nº de linhas removidas em cada passo).
funil = pd.DataFrame(contagem, columns=["passo", "linhas"])
funil["removidas"] = (-funil["linhas"].diff()).fillna(0).astype(int)
print(f"Gravado: {FICHEIRO_LIMPO.relative_to(FICHEIRO_LIMPO.parents[2])}  "
      f"({len(df)} linhas × {df.shape[1]} colunas)\n")
funil

Gravado: data/processed/dados_limpos.csv  (1623 linhas × 23 colunas)



,passo,linhas,removidas
0,Exportação original,1661,0
1,Só apartamentos à venda,1661,0
2,Sem valores impossíveis,1652,9
3,Sem duplicados,1623,29


## Conclusões

- Partimos de uma exportação com **mais de 450 colunas** e ficámos com uma tabela de **23 colunas** com o essencial.
- Removemos **dados pessoais**, **valores impossíveis** e **duplicados** — no total, apenas ~2% dos anúncios, o que indica que a exportação tinha boa qualidade.
- Os campos com mais informação em falta são os **extras** (piscina, jardim, terraço, arrecadação ~40%). Ficaram vazios de propósito; decidimos como tratá-los na modelação.
- **Limitação importante:** não existe nenhuma coluna com a **idade do imóvel** nem com o **certificado energético**, que seriam úteis para prever o preço.

➡️ **Próximo passo:** `02_analise_exploratoria.ipynb`